In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torch
import torchvision.transforms.functional as TF

In [ ]:
IMG_SIZE = (224,224)
class XBDSemiSupervisedDataset(Dataset):
    def __init__(self, dataframe, labeled=True):
        self.df = dataframe
        self.labeled = labeled

    def load_img(self, path):
        img = Image.open(path).convert("RGB").resize(IMG_SIZE)
        img = np.array(img).astype(np.float32) / 255.0
        return img

    def load_mask(self, path):
        mask = Image.open(path).convert("L").resize(IMG_SIZE)
        mask = np.array(mask).astype(np.uint8)
        return mask

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pre = self.load_img(row["pre"])
        post = self.load_img(row["post"])

        pre_t = TF.to_tensor(Image.fromarray((pre*255).astype(np.uint8)))
        post_t = TF.to_tensor(Image.fromarray((post*255).astype(np.uint8)))

        inp = torch.cat([pre_t, post_t], dim=0)   # 6-channel

        if self.labeled:
            mask = self.load_mask(row["mask"])
            mask = torch.from_numpy(mask).long()
            return inp, mask
        else:
            return inp


In [ ]:
df = pd.read_csv("splits/train_multi_v2.csv")
labeled_df, unlabeled_df = train_test_split(
    df,
    test_size=0.90,
    random_state=42
)

print("Labeled samples:", len(labeled_df))
print("Unlabeled samples:", len(unlabeled_df))


Labeled samples: 75
Unlabeled samples: 680


In [ ]:
labeled_ds = XBDSemiSupervisedDataset(labeled_df, labeled=True)
unlabeled_ds = XBDSemiSupervisedDataset(unlabeled_df, labeled=False)

labeled_loader = DataLoader(
    labeled_ds,
    batch_size=4,
    shuffle=True,
    num_workers=2
)

unlabeled_loader = DataLoader(
    unlabeled_ds,
    batch_size=4,
    shuffle=True,
    num_workers=2
)